# Graph here, graph there

Here is one way of getting the solution. There are others, feel free to
share yours.

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

from libpysal import graph

In [ ]:
suburbanisation = gpd.read_file("/Users/martinfleis/Downloads/zony_suburbanizace_2008_2016_8970316512140498173.zip")

The `obec_kod` column is not unique because when a municipality is not
contiguous, it is split. Merge them together to get correct boundaries
per municipality.

In [ ]:
suburbanisation = suburbanisation.dissolve("obec_kod")

This has already set the `obec_kod` as an index, so we can use it.

Create a contiguity matrix using the queen criterion

In [ ]:
queen = graph.Graph.build_contiguity(suburbanisation, rook=False)

Let’s focus on Prague (554782 in the table). How many neighbours does it
have?

One option:

In [ ]:
len(queen[554782])

Other option:

In [ ]:
queen.cardinalities[554782]

Reproduce the previous section’s zoom plot with Prague and its
neighbours. Can you make that plot as both static and interactive maps?

In [ ]:
m = suburbanisation.loc[queen[554782].index].explore(color="#25b497")
suburbanisation.loc[[554782]].explore(m=m, color="#fa94a5")

With `matplotlib`:

In [ ]:
ax = suburbanisation.loc[queen[554782].index].plot(color="#25b497")
suburbanisation.loc[[554782]].plot(ax=ax, color="#fa94a5")

Create a block spatial weights matrix where every geometry is connected
to other geometries in the NUTS2 region. For that, use the NUTS2_kod
column of the table.

In [ ]:
nuts = graph.Graph.build_block_contiguity(suburbanisation["NUTS2_kod"])

Create KNN weights with 5 neighbours. Remember that KNN expects point
geometry.

In [ ]:
knn5 = graph.Graph.build_knn(suburbanisation.set_geometry(suburbanisation.centroid), k=5)

Compare the number of neighbours by geometry for the three weights
matrices. Which one has more? Why?

For Prague only:

In [ ]:
knn5.cardinalities[554782]

In [ ]:
nuts.cardinalities[554782]

Prague is a NUTS2 region by itself. See the data plotted by NUTS2:

In [ ]:
suburbanisation.plot("NUTS2_kod")

For whole data frame

In [ ]:
nuts.cardinalities.mean()

In [ ]:
knn5.cardinalities.mean()

Let’s have a look at spatial lag. Before proceeding, you will probably
need to pre-process the column with the population (“obyv_31122”) since
it comes as string. Assuming the GeoDataFrame is called suburbanisation,
you can do the following to cast it to float.

In [ ]:
suburbanisation["obyv_31122"] = (
    suburbanisation["obyv_31122"].replace("?", None).astype(float)
)

Measure spatial lag (as mean, so don’t forget to standardise your
weights) of the “obyv_31122” column using all weights matrices you have
created.

In [ ]:
queen_r = queen.transform("R")
nuts_r = nuts.transform("R")
knn_r = knn5.transform("R")

# Prague is too extreme, let's log it
suburbanisation["obyv_log"] = np.log(suburbanisation["obyv_31122"])

suburbanisation["queen_lag"] = queen_r.lag(suburbanisation["obyv_log"])
suburbanisation["nuts_lag"] = nuts_r.lag(suburbanisation["obyv_log"])
suburbanisation["knn_lag"] = knn_r.lag(suburbanisation["obyv_log"])

What is the difference in results for Prague? Do you know why?

In [ ]:
suburbanisation.loc[554782, ["queen_lag", "nuts_lag", "knn_lag"]]